# OCR comprobantes con PaddleOCR

Supuestamente maneja una mayor precisión en textos con layouts complejos. No requiere instalación de binario externo como con tesseract. Devuelve bounding boxes con coordenadas y nivel de confianza por detección.

### Dependencias
PaddleOCR requiere PaddlePaddle como backend. Se instala todo desde pip, sin binarios externos.

In [22]:
# !pip install paddlepaddle==2.6.2
# !pip install paddleocr==2.9.1

In [23]:
# utilidades
# !pip install pymupdf pillow opencv-python-headless matplotlib setuptools

### Imports e intancia de PaddleOCR

La primera ejecución descarga los modelos automáticamente y quedan cacheados en `~/.paddleocr/`.

In [24]:
# from zmq import backend
import cv2
import numpy as np
import fitz
import re
from paddleocr import PaddleOCR
from PIL import Image
from pathlib import Path

# Config OCR 
# lang='es' para español (incluye modelo latino)
ocr_engine = PaddleOCR(
    use_textline_orientation=True,
    lang='es',
    #backend='onnxruntime'
)

print('PaddleOCR inicializado correctamente.')

[2026/04/18 08:08:29] ppocr DEBUG: Namespace(help='==SUPPRESS==', use_gpu=False, use_xpu=False, use_npu=False, use_mlu=False, ir_optim=True, use_tensorrt=False, min_subgraph_size=15, precision='fp32', gpu_mem=500, gpu_id=0, image_dir=None, page_num=0, det_algorithm='DB', det_model_dir='C:\\Users\\Admin/.paddleocr/whl\\det\\en\\en_PP-OCRv3_det_infer', det_limit_side_len=960, det_limit_type='max', det_box_type='quad', det_db_thresh=0.3, det_db_box_thresh=0.6, det_db_unclip_ratio=1.5, max_batch_size=10, use_dilation=False, det_db_score_mode='fast', det_east_score_thresh=0.8, det_east_cover_thresh=0.1, det_east_nms_thresh=0.2, det_sast_score_thresh=0.5, det_sast_nms_thresh=0.2, det_pse_thresh=0, det_pse_box_thresh=0.85, det_pse_min_area=16, det_pse_scale=1, scales=[8, 16, 32], alpha=1.0, beta=1.0, fourier_degree=5, rec_algorithm='SVTR_LCNet', rec_model_dir='C:\\Users\\Admin/.paddleocr/whl\\rec\\latin\\latin_PP-OCRv3_rec_infer', rec_image_inverse=True, rec_image_shape='3, 48, 320', rec_batc

## 3. Preprocesamiento de imagen

PaddleOCR es más robusto que Tesseract ante imágenes "sucias", pero el preprocesamiento
sigue mejorando resultados en capturas de pantalla o fotos de celular con baja resolución.

In [25]:
def preprocess_image(img_bgr: np.ndarray, min_ancho: int = 1000) -> np.ndarray:
    """
    Preprocesamiento para OCR trabajando únicamente en BGR (OpenCV).
    """
    # reescala de la imagen
    h, w = img_bgr.shape[:2]
    if w < min_ancho:
        escala = min_ancho / w
        img_bgr = cv2.resize(
            img_bgr,
            None,
            fx=escala,
            fy=escala,
            interpolation=cv2.INTER_CUBIC
        )

    # Reducción de ruido leve
    img_bgr = cv2.fastNlMeansDenoisingColored(
        img_bgr,
        None,
        h=6,
        hColor=6
    )

    return img_bgr

## 4. Conversión de PDF a imágenes
Cada página del PDF se rasteriza a alta resolución.

In [26]:
def pdf_to_img(pdf_path: str) -> list:
    doc = fitz.open(pdf_path)
    images = []

    for page in doc:
        pix = page.get_pixmap(dpi=300)
        img = np.frombuffer(pix.samples, dtype=np.uint8)
        img = img.reshape(pix.height, pix.width, pix.n)

        if pix.n == 4:
            img = cv2.cvtColor(img, cv2.COLOR_BGRA2BGR)
        else:
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)

        images.append(img)

    doc.close()
    return images

def load_img(path_image: str) -> np.ndarray:
    path = Path(path_image)

    if not path.exists():
        raise FileNotFoundError(f"No se encontró el archivo: {path_image}")
    
    img = cv2.imread(str(path))
    
    if img is None:
        raise ValueError(f"No se pudo cargar la imagen: {path_image}")

    return img

## 5. Extracción de texto con PaddleOCR

PaddleOCR devuelve **bloques estructurados** con:
- bbox: coordenadas del bounding box [[x1,y1],[x2,y2],[x3,y3],[x4,y4]]
- texto: string detectado
- confianza: score de 0.0 a 1.0

In [27]:
def extract_text(img_bgr: np.ndarray,
                         confidence_threshold: float = 0.6) -> tuple:
    """
    Compatible con PaddleOCR 2.x.

    Retorna:
      - texto_plano (str)
      - blocks (list)
    """
    result = ocr_engine.ocr(img_bgr, cls=True)

    blocks = []
    lines = []

    if not result:
        return "", []

    for line in result[0]:
        bbox = line[0]
        text = line[1][0]
        confidence = float(line[1][1])

        if confidence >= confidence_threshold:
            blocks.append({
                "bbox": bbox,
                "text": text,
                "confidence": round(confidence, 3)
            })

            lines.append(text)

    return "\n".join(lines), blocks


def process_file(path: str,
                 preprocess: bool = True,
                 confidence_threshold: float = 0.6) -> dict:
    ext = Path(path).suffix.lower()
    result = {
        "file": path,
        "pages": []
    }

    if ext == ".pdf":
        images = pdf_to_img(path)
    elif ext in (".png", ".jpg", ".jpeg", ".tiff", ".bmp", ".webp"):
        images = [load_img(path)]
    else:
        raise ValueError(f"Formato no soportado: {ext}")

    for i, img_bgr in enumerate(images, start=1):
        if preprocess:
            img_bgr = preprocess_image(img_bgr)
        text, blocks = extract_text(img_bgr, confidence_threshold)

        result["pages"].append({
            "nr": i,
            "text": text,
            "blocks": blocks
        })
        print(f"✔ Página {i}: {len(blocks)} blocks detectados")

    return result

## 6. Extracción de campos clave del comprobante

In [28]:
PATRONES = {
    'importe':            r'\$\s?([\d.,]+)|importe[:\s]+([\d.,]+)',
    'fecha':              r'(\d{1,2}[/-]\d{1,2}[/-]\d{2,4})',
    'hora':               r'(\d{2}:\d{2}(?::\d{2})?)',
    'cbu_cvu':            r'(?:CBU|CVU)[:\s]*([\d]{22})',
    'alias':              r'[Aa]lias[:\s]+([\w.]+)',
    'cuit_cuil':          r'(?:CUIT|CUIL)[:\s]*(\d{2}-?\d{8}-?\d)',
    'numero_comprobante': r'(?:N[°º]?|Comprobante|Nro\.?)[:\s#]*(\d+)',
    'banco_origen':       r'(?:Banco|Entidad)[:\s]+([A-Za-záéíóúñÁÉÍÓÚÑ ]+)',
}

def get_fields(text: str) -> dict:
    """
    Aplica regex sobre el textoOCR para extraer campos estructurados.
    """
    fields = {}
    for field, pattern in PATRONES.items():
        match = re.search(pattern, text, re.IGNORECASE)
        if match:
            value = next((g for g in match.groups() if g), None)
            fields[field] = value.strip() if value else None
        else:
            fields[field] = None
    return fields

## 7. Visualización de resultados
Muestra texto crudo, campos parseados, y opcionalmente los bloques con sus scores de confianza.

In [32]:
def show_result(result: dict,
                      show_raw_text: bool = True,
                      show_blocks: bool = False):
    """
    Imprime de forma legible el texto OCR, los bloques y los campos extraídos.
    `show_blocks=True` muestra cada detección con su bbox y confianza.
    """
    print(f"{'='*60}")
    print(f"File: {result['file']}")
    print(f"Processed pages: {len(result['pages'])}")

    for page in result['pages']:
        print(f"\n--- Página {page['nr']} ({len(page['blocks'])} blocks) ---")

        if show_raw_text:
            print("\n[TEXTO CRUDO OCR]")
            print(page['text'])

        if show_blocks:
            print("\n[BLOQUES DETALLADOS]")
            for b in page['blocks']:
                print(f"  [{b['confidence']:.2f}] {b['text']}")

        fields = get_fields(page['text'])
        print("\n[CAMPOS EXTRAÍDOS]")
        for field, value in fields.items():
            state = '✔' if value else '✗'
            print(f"  {state} {field:<25}: {value or 'no detectado'}")

    print(f"{'='*60}")

## 8. Visualización de bounding boxes sobre la imagen
Dibuja los bloques detectados directamente sobre la imagen — útil para diagnosticar detecciones erróneas.

In [30]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

def visualizar_detecciones(img_bgr: np.ndarray, bloques: list,
                            titulo: str = 'Detecciones PaddleOCR'):
    """
    Dibuja los bounding boxes de cada bloque sobre la imagen original.
    El color del recuadro va de rojo (baja confianza) a verde (alta confianza).
    """
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    fig, ax = plt.subplots(1, 1, figsize=(14, 18))
    ax.imshow(img_rgb)
    ax.set_title(titulo, fontsize=12)
    ax.axis('off')

    for bloque in bloques:
        pts = np.array(bloque['bbox'], dtype=np.int32)
        conf = bloque['confianza']
        # Verde = alta confianza, Rojo = baja
        color = (0, conf, 1 - conf)  # RGB relativo

        # Dibujar polígono
        poly = patches.Polygon(pts, closed=True,
                               edgecolor=color, linewidth=1.5, fill=False)
        ax.add_patch(poly)

        # Etiqueta con texto y confianza
        x, y = pts[0]
        ax.text(x, y - 4, f"{bloque['texto']} ({conf:.2f})",
                fontsize=6, color='white',
                bbox=dict(facecolor='black', alpha=0.5, pad=1))

    plt.tight_layout()
    plt.show()

## 9. Ejecución principal

In [33]:
FILES = [
    './comprobantes/c1.pdf',
    './comprobantes/img1.jpeg'
]

results = []

for file in FILES:
    print(f'\nProcesando: {file}')
    try:
        result = process_file(file, preprocess=True, confidence_threshold=0.6)
        results.append(result)
        show_result(result, show_raw_text=True, show_blocks=False)
    except FileNotFoundError as e:
        print(f'  ⚠ Archivo no encontrado: {e}')
        raise
    except Exception as e:
        print(f'  ✗ Error procesando {file}: {e}')
        raise


Procesando: ./comprobantes/c1.pdf
[2026/04/18 08:10:00] ppocr WARNING: Since the angle classifier is not initialized, it will not be used during the forward process
[2026/04/18 08:10:00] ppocr DEBUG: dt_boxes num : 17, elapsed : 0.07081747055053711
[2026/04/18 08:10:00] ppocr DEBUG: rec_res num  : 17, elapsed : 0.2221982479095459
✔ Página 1: 16 blocks detectados
File: ./comprobantes/c1.pdf
Processed pages: 1

--- Página 1 (16 blocks) ---

[TEXTO CRUDO OCR]
ICBC
Comprobante de transferencia
13:18 - 09 de Febrero, 2026
N de operación: 1523
Transferiste
$100.000,00
a Fernando Mario Romero Munoz
CUIT/CUIL: 24-37366211-5
BRUBANK
CBU/CVU: 1430001713031104620012
Alias: fer.rom.mu.bru
De Fernando Mario Romero Munoz
Cuenta: CA $ ...71/51
Concepto: Varios
Sujeto a impuestos y comisiones determinadas por tu banco
La transferencia se cursó al destino de forma inmediata

[CAMPOS EXTRAÍDOS]
  ✔ importe                  : 100.000,00
  ✗ fecha                    : no detectado
  ✔ hora               